# 프롬프트 실험실 (Prompt Lab)

운영 파이프라인과 완전히 분리된 노트북입니다. GitHub에 아무것도 push하지 않습니다.

사용 방법:
1. 런타임 → 런타임 유형 변경 → T4 GPU 선택
2. 셀 1, 2를 한 번만 실행 (모델 로드, 1~2분)
3. 셀 3의 `PROMPT`, `NEGATIVE_PROMPT`, `STEPS`, `GUIDANCE` 값을 자유롭게 바꿔가며
   반복 실행 → 결과를 즉시 확인 (모델은 이미 로드되어 있어 매번 1초 내)
4. 마음에 드는 조합을 찾으면 그 `PROMPT` 문구를 그대로 복사해서 전달해주세요.
   (Claude가 prompt_templates.json에 반영합니다)

In [ ]:
# 1. 환경 설정
!pip install diffusers transformers accelerate -q

import torch
from diffusers import AutoPipelineForText2Image

print("환경 설정 완료")

In [ ]:
# 2. 모델 로드 (1~2분, 한 번만 실행)
pipe = AutoPipelineForText2Image.from_pretrained(
    "stabilityai/sdxl-turbo", torch_dtype=torch.float16, variant="fp16"
).to("cuda")
print("모델 로드 완료")

In [ ]:
# 3. 실험 셀 - 아래 값들을 자유롭게 바꿔가며 반복 실행하세요

PROMPT = (
    "A single bowl of Korean bibimbap with rice, fried egg, and "
    "colorful vegetables, top-down view, one dish only, no other "
    "plates or bowls in frame, professional studio food photography, "
    "bright even lighting, isolated on a pure white background"
)

# guidance_scale > 0 이어야 negative_prompt가 적용됩니다.
# SDXL-Turbo 권장값은 0이지만, 1.0~2.0 정도면 속도는 거의 그대로면서
# negative_prompt로 "여러 그릇" 같은 패턴을 억제할 수 있습니다.
NEGATIVE_PROMPT = (
    "multiple bowls, many small dishes, grid of plates, banchan spread, "
    "tiled composition, repeated pattern, collage"
)

STEPS = 4
GUIDANCE = 1.5
NUM_IMAGES = 4  # 한 번에 몇 장 비교할지

# ─────────────────────────────────────────────
for i in range(NUM_IMAGES):
    seed = torch.randint(0, 2**31 - 1, (1,)).item()
    generator = torch.Generator(device="cuda").manual_seed(seed)
    image = pipe(
        prompt=PROMPT,
        negative_prompt=NEGATIVE_PROMPT,
        num_inference_steps=STEPS,
        guidance_scale=GUIDANCE,
        width=1024,
        height=1024,
        generator=generator,
    ).images[0]
    print(f"seed={seed}")
    display(image)